# Distribuzioni di carica per canale, da una run o da una lista

Carica dei cinque canali del LNGS -- SiPM ch16-18, PMT ch20-21 -- dalle run della DAQ: una sola, o
tante, sommate e confrontate fra loro. Cosa leggere si sceglie nella cella **Scelta delle run**; il
resto si esegue com'e'.

**Il metodo** e' quello di `tools/carica_padme.py`, lo strumento delle analisi di fine ottobre:

- **cancello e piedistallo automatici, per canale**, con la regola del monitor: la forma d'onda
  media del 20% di eventi piu' alti dice dove l'impulso supera il 10% del picco; il cancello va da
  10 ns prima a 20 ns dopo, il piedistallo da 10 ns fino a 20 ns prima del cancello. Si misurano
  **sulla prima run della lista** e valgono per tutte: cariche integrate in cancelli diversi non si
  sommano. Un cancello scritto a mano in `CANCELLO` vince;
- **le analisi della 0011 e della 0013 non usavano i cancelli automatici**: cancello 240-500 ns e
  piedistallo 0-150 ns, fissi per tutti i canali, ed e' con quelli che e' stato misurato
  K = 1647 ADC·ns/PE. Il cancello automatico dei PMT e' piu' stretto (~240-320 ns) e lascia fuori
  la luce lenta: per confrontarsi con quelle analisi c'e' una riga pronta nella cella dei parametri;
- **carica = somma dei campioni nel cancello meno il piedistallo, per il passo di campionamento**,
  in ADC·ns (1 pC = 204.8 ADC·ns), col segno del verso del canale: positiva anche per impulsi
  negativi. Il verso lo dice il DCOffset scritto nel log della run;
- **piedistallo per evento o per blocco.** Per evento e' la media della finestra in quell'evento,
  come nel monitor e in `carica_padme.py`. Per blocco e' la mediana di quelle medie su
  `EVENTI_PER_BLOCCO` eventi consecutivi (5000, ~90 s): segue le derive lente e non risente degli
  impulsi al buio che cadono nella finestra. **Di serie per blocco sui SiPM**: con un cancello lungo
  la sottrazione evento per evento allarga la carica (SiPM a 51 V: sigma 1500-2000 ADC·ns contro
  ~600 con la base fissa del canale, analisi del 2026-10-28), e un impulso al buio nella finestra
  porta la carica a valori negativi. Si calcolano tutte e due: il grafico mostra quella scelta e,
  in grigio, l'altra, con la larghezza del picco degli eventi senza impulso;
- **senza correzione PADME**, che sulla carica conta poco (stessa analisi): per quella c'e'
  `tools/carica_padme.py`;
- un evento con un campione al fondo scala nel cancello e' **saturo**: la sua carica e' un limite
  inferiore, e nel grafico e' a parte.

**Dove gira.** Sul PC DAQ, dove stanno i dati. Sul Mac col kernel `jupyter-global` (ha `h5py`), coi
file copiati in una cartella da scrivere in `CARTELLA`: col `.h5.gz` serve anche il `.log`, che porta
DCOffset e durata della run.

**Cosa scrive.** I grafici e un `.json` che dice come sono state fatte le cariche in
`measurements/carica_run_<run>*`; le cariche evento per evento in `data/carica/carica_run_<run>.npz`,
fuori da git perche' pesano ~3 MB per run. Le run gia' lette restano in `data/carica/run/`:
rieseguendo con una lista piu' lunga si leggono solo quelle nuove. Per rileggere le cariche:

```python
z = np.load("data/carica/carica_run_....npz")
q = z["q_blocco_ch16"]; meta = json.loads(str(z["meta"]))
```

In [ ]:
## === Import ===
import contextlib
import glob
import json
import os
import re
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

# La radice del repository si cerca risalendo dalla cartella del notebook:
# cosi' funziona sul PC DAQ e sul Mac, dovunque stia la copia.
ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "tools", "daqio.py")):
    if os.path.dirname(ROOT) == ROOT:
        raise RuntimeError("Non trovo la radice del repository (tools/daqio.py) risalendo da "
                           + os.getcwd())
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "tools"))

import daqio          # lettura dei file, anche .gz e run in corso
import efficienza     # verso degli impulsi dal log della run
import carica_padme   # cancello e piedistallo automatici, gli stessi delle analisi di ottobre

In [ ]:
# ============================================================
# SCELTA DELLE RUN
# ============================================================
# Dove cercare le run date per numero, per nome o con un glob. Un percorso
# completo vale da qualunque cartella.
CARTELLA = os.path.join(ROOT, "data", "LNGS")

# Il numero di run riparte per ogni prefisso, e il prefisso cambia con la
# tensione letta dall'alimentatore (54-0V, 54-1V, ...): un numero da solo e'
# ambiguo, il prefisso lo fissa. Vale solo per le run date per numero.
PREFISSO = "LNGS__54-0V_"

# Una run:           RUN = 158
# Un intervallo:     RUN = range(150, 159)          # dalla 0150 alla 0158
# Una lista:         RUN = [150, 152, 158]
# Per nome:          RUN = ["LNGS__54-0V_0158_750Ms_55PT", "LNGS__54-1V_0000_750Ms_55PT"]
# Per percorso:      RUN = ["/percorso/qualunque/LNGS__54-0V_0158_750Ms_55PT.h5.gz"]
# Con un glob:       RUN = "LNGS__54-0V_015?_750Ms_55PT"
# Le forme si mescolano: RUN = [150, 151, "LNGS__54-1V_0000_750Ms_55PT"]
RUN = 158

In [ ]:
# ============================================================
# PARAMETRI
# ============================================================
CANALI = [16, 17, 18, 20, 21]
RIVELATORE = {16: "SiPM", 17: "SiPM", 18: "SiPM", 20: "PMT", 21: "PMT"}

# Cancello scritto a mano, in ns, per i canali dove quello automatico non
# va: CANCELLO = {20: (240, 340)}. La finestra del piedistallo lo segue da
# sola (da 10 ns fino a 20 ns prima del cancello); FINESTRA_PED la impone.
CANCELLO = {}
FINESTRA_PED = {}
# La ricetta delle analisi della 0011 e della 0013 (spettri del fondo, K = 1647 ADC*ns/PE):
# CANCELLO = {c: (240, 500) for c in CANALI}; FINESTRA_PED = {c: (0, 150) for c in CANALI}

# Il verso dell'impulso lo dice il log della run; qui lo si impone (+1 / -1).
VERSO = {}

# "evento" o "blocco" (vedi l'introduzione); i canali non elencati: "evento".
PIEDISTALLO = {16: "blocco", 17: "blocco", 18: "blocco", 20: "evento", 21: "evento"}
EVENTI_PER_BLOCCO = 5000

EVENTI_PER_FINESTRE = 5000    # eventi della prima run su cui si misurano i cancelli
MAX_EVENTI_PER_RUN = None     # None = tutti; un numero per un'occhiata veloce

USCITA = os.path.join(ROOT, "measurements")    # grafici e descrizione (.json)
DATI = os.path.join(ROOT, "data", "carica")     # cariche evento per evento (.npz), fuori da git
CACHE = os.path.join(DATI, "run")               # le run gia' lette

In [ ]:
## === Funzioni ===
TTT_NS = 17.07       # passo del Trigger Time Tag, misurato il 2026-10-23 (CLAUDE.md)
TTT_GIRO = 2 ** 31   # contatore a 31 bit: un giro ogni ~36.7 s
FONDO_SCALA = 4095   # 12 bit
VERSIONE_CACHE = 1   # da cambiare se cambia quello che va in cache


def _dati(percorsi):
    """Un file di dati per run. Col .h5 e il .h5.gz insieme la run e' appena
    stata compressa e il .h5 sta per sparire: vale il .gz, che archivia.py
    rinomina col suo nome solo quando e' completo e verificato."""
    per_run = {}
    for p in percorsi:
        if p.endswith((".h5", ".h5.gz")):
            n = efficienza.nome_run(p)
            if n not in per_run or p.endswith(".gz"):
                per_run[n] = p
    return [per_run[n] for n in sorted(per_run)]


def trova_file(scelta):
    """I file di dati della scelta RUN, nell'ordine in cui sono dati."""
    if isinstance(scelta, (str, int, np.integer)):
        scelta = [scelta]
    fuori = []
    for s in scelta:
        if isinstance(s, (int, np.integer)):
            trovati = _dati(glob.glob(os.path.join(CARTELLA, "%s%04d_*" % (PREFISSO, s))))
            if len(trovati) > 1:
                raise ValueError("Run %d con prefisso %r: piu' file (%s). Darla per nome."
                                 % (s, PREFISSO, ", ".join(map(os.path.basename, trovati))))
        elif os.path.isfile(s):
            trovati = [s]
        else:
            trovati = _dati(glob.glob(os.path.join(CARTELLA, s))
                            + glob.glob(os.path.join(CARTELLA, s + ".h5*")))
        if not trovati:
            raise FileNotFoundError("Nessun file di dati per %r in %s" % (s, CARTELLA))
        fuori += [p for p in trovati if p not in fuori]
    return fuori


def etichette(nomi):
    """La parte dei nomi che cambia da una run all'altra, tagliata ai '_':
    '0150', '0151'... o '54-0V_0158', '54-1V_0000' se cambia il prefisso.
    Ritorna (prefisso comune, etichette, suffisso comune)."""
    if len(nomi) == 1:
        return "", list(nomi), ""
    p = os.path.commonprefix(nomi)
    p = p[:p.rfind("_") + 1]
    s = os.path.commonprefix([n[::-1] for n in nomi])[::-1]
    s = s[s.find("_"):] if "_" in s else ""
    return p, [n[len(p):len(n) - len(s)] for n in nomi], s


def leggi_log(path):
    """Dal log della run: verso per canale, durata, eventi, collegamento perso,
    correzione DRS4. None se il log non c'e' (un .h5.gz copiato da solo)."""
    log = re.sub(r"\.h5(\.gz)?$", ".log", path)
    try:
        testo = open(log, errors="replace").read()
    except OSError:
        return None
    t = re.search(r"Acquisition time: ([0-9.]+)", testo)
    n = re.search(r"Events written to file: (\d+)", testo)
    return {"versi": efficienza.polarita_dal_log(log),
            # senza riepilogo la run e' in corso, o e' morta prima di scriverlo
            "durata_s": float(t.group(1)) if t else None,
            "eventi": int(n.group(1)) if n else None,
            "collegamento_perso": "giving up" in testo,
            "drs4": ("on" if "DRS4 corrections ENABLED" in testo else
                     "off" if re.search(r"DRS4 corrections (DISABLED|disabled)", testo) else "?")}


@contextlib.contextmanager
def apri(path):
    """(file h5py, header). Un .gz si decomprime in un file temporaneo, che si
    cancella all'uscita. Una run in corso, o chiusa male, si apre in SWMR."""
    try:
        f, tmp = daqio.open_file(path)
    except daqio.DaqFileError:
        f, tmp = daqio.open_file(path, live=True)
    try:
        if "/events/waveforms" not in f:
            raise daqio.DaqFileError("%s: formato v1 (un dataset per evento), non letto qui"
                                     % os.path.basename(path))
        yield f, daqio.read_header(f)
    finally:
        f.close()
        daqio._cleanup(tmp)


def geometria(hdr):
    """(campioni per canale, passo in ns, {canale: colonne della riga})."""
    ns = int(hdr["SamplesPerChannel"])
    if "SamplingTime" in hdr:
        dt = float(hdr["SamplingTime"]) * 1e9
    else:  # file vecchi: '750MHz', '1GHz'
        m = re.match(r"([0-9.]+)\s*([GM])", str(hdr["SamplingRate"]))
        dt = 1000.0 / (float(m.group(1)) * (1000 if m.group(2) == "G" else 1))
    nel_file = [int(c) for c in hdr["ChannelList"]]
    mancanti = [c for c in CANALI if c not in nel_file]
    if mancanti:
        raise ValueError("Canali non presenti nel file: %s (ci sono %s)" % (mancanti, nel_file))
    return ns, dt, {c: slice(nel_file.index(c) * ns, (nel_file.index(c) + 1) * ns) for c in CANALI}


def stessa_presa(hdr, rif, nome):
    """Con frequenza, campioni o PostTrigger diversi l'impulso cade altrove, e
    i cancelli della prima run non valgono: la run va analizzata a parte."""
    for k in ("SamplingRate", "SamplesPerChannel", "PostTriggerSize"):
        if k in rif and str(hdr.get(k)) != str(rif[k]):
            raise ValueError("%s: %s = %s, nella prima run %s. Toglierla dalla lista, o "
                             "analizzarla a parte." % (nome, k, hdr.get(k), rif[k]))


def verso_canale(c, info, y):
    """+1 / -1, e chi l'ha deciso: a mano, il log, o -- senza log -- la
    posizione del piedistallo, che e' cio' che il DCOffset decide: in basso
    per impulsi positivi, in alto per negativi."""
    if c in VERSO:
        return VERSO[c], "a mano"
    v = info["versi"] if info else None
    if isinstance(v, dict) and v.get(c) in (1, -1):
        return v[c], "dal DCOffset nel log"
    if v in (1, -1):
        return v, "da TriggerPolarity nel log"
    base = float(np.median(y))
    return (1 if base < 2048 else -1), "dal piedistallo a %.0f conteggi: log assente" % base


def controlla_versi(info, nome):
    """Il verso di un canale non deve cambiare fra le run di una lista."""
    if info is None or info["versi"] is None:
        return
    for c in CANALI:
        v = info["versi"].get(c) if isinstance(info["versi"], dict) else info["versi"]
        if c not in VERSO and v in (1, -1) and v != VERSI[c]:
            raise ValueError("%s: ch%d ha impulsi %s, nella prima run %s. Le run di una lista "
                             "devono avere gli stessi DCOffset." % (
                                 nome, c, "positivi" if v > 0 else "negativi",
                                 "positivi" if VERSI[c] > 0 else "negativi"))


def leggi_campione(path, n):
    """Header, primi n eventi dei canali analizzati (eventi, canali, campioni) e passo in ns."""
    with apri(path) as (f, hdr):
        _, dt, col = geometria(hdr)
        ds = f["/events/waveforms"]
        w = np.asarray(ds[:min(n, ds.shape[0])])
    return hdr, np.stack([w[:, col[c]] for c in CANALI], axis=1), dt


def finestre_canale(c, y, segno, t_ns):
    """Cancello (ns) e piedistallo (indici) di un canale, e chi ha deciso il cancello."""
    if c in CANCELLO:
        canc, come = tuple(float(x) for x in CANCELLO[c]), "by hand"
    else:
        canc, come = carica_padme.finestre(y.astype(np.float32), segno, t_ns), "automatic"
        if canc is None:
            raise RuntimeError(
                "ch%d: nessun impulso riconoscibile nella forma d'onda media dei primi %d eventi. "
                "Dare il cancello a mano, CANCELLO[%d] = (da_ns, a_ns), o togliere il canale da "
                "CANALI." % (c, len(y), c))
    if c in FINESTRA_PED:
        pi = tuple(int(np.searchsorted(t_ns, x)) for x in FINESTRA_PED[c])
    else:
        pi = tuple(int(i) for i in carica_padme.indici_ped(t_ns, canc))
    if pi[1] - pi[0] < carica_padme.MIN_CAMPIONI_PED:
        raise RuntimeError("ch%d: cancello a %.0f ns, troppo presto per un piedistallo prima: "
                           "darlo a mano in FINESTRA_PED[%d]" % (c, canc[0], c))
    gi = tuple(int(np.searchsorted(t_ns, x)) for x in canc)
    return {"cancello_ns": canc, "gi": gi, "pi": pi, "come": come}


def leggi_run(path, info, fin, rif):
    """Per ogni canale, evento per evento: somma dei campioni nel cancello,
    media della finestra del piedistallo, saturazione. Letta a blocchi di
    ~50 MB: una run da 100000 eventi tutta insieme sarebbe 1 GB.

    In cache vanno solo le run chiuse, sotto la chiave delle finestre:
    cambiato un cancello, la run si rilegge. Ritorna (risultato, dalla_cache)."""
    nome = efficienza.nome_run(path)
    chiave = json.dumps({"versione": VERSIONE_CACHE, "max": MAX_EVENTI_PER_RUN,
                         "finestre": {str(c): [fin[c]["gi"], fin[c]["pi"]] for c in CANALI}},
                        sort_keys=True)
    cache = os.path.join(CACHE, nome + ".npz")
    # Un .gz e' sempre una run chiusa: archivia.py comprime solo quelle.
    chiusa = path.endswith(".gz") or (info is not None and info["durata_s"] is not None)
    if chiusa and os.path.exists(cache):
        with np.load(cache) as z:
            if str(z["chiave"]) == chiave:
                r = {k: z[k] for k in z.files if k != "chiave"}
                r["n"], r["n_file"] = int(r["n"]), int(r["n_file"])
                return r, True
    with apri(path) as (f, hdr):
        stessa_presa(hdr, rif, nome)
        _, _, col = geometria(hdr)
        ds = f["/events/waveforms"]
        # A run in corso i dataset possono differire di qualche riga
        n_file = min([ds.shape[0]] + [f[k].shape[0] for k in
                                      ("/events/event_counter", "/events/trigger_time_tag") if k in f])
        n = min(n_file, MAX_EVENTI_PER_RUN or n_file)
        r = {"n": n, "n_file": n_file}
        for c in CANALI:
            r["S_ch%d" % c] = np.empty(n)
            r["P_ch%d" % c] = np.empty(n, np.float32)
            r["sat_ch%d" % c] = np.empty(n, bool)
        passo = max(256, int(5e7 // (2 * ds.shape[1])))
        for a in range(0, n, passo):
            b = min(n, a + passo)
            w = np.asarray(ds[a:b])
            for c in CANALI:
                y = w[:, col[c]]
                (g0, g1), (p0, p1) = fin[c]["gi"], fin[c]["pi"]
                yg = y[:, g0:g1]
                r["S_ch%d" % c][a:b] = yg.sum(axis=1, dtype=np.float64)
                r["P_ch%d" % c][a:b] = y[:, p0:p1].mean(axis=1, dtype=np.float64)
                r["sat_ch%d" % c][a:b] = (yg.max(axis=1) >= FONDO_SCALA) | (yg.min(axis=1) <= 0)
        for k, d in (("event_counter", "/events/event_counter"), ("ttt", "/events/trigger_time_tag")):
            r[k] = np.asarray(f[d][:n]) if d in f else np.zeros(0, np.uint32)
    if chiusa:
        os.makedirs(CACHE, exist_ok=True)
        np.savez_compressed(cache + ".tmp.npz", chiave=chiave, **r)
        os.replace(cache + ".tmp.npz", cache)
    return r, False


def durata(r, info):
    """Tempo di acquisizione in s, e da dove viene: quello che la DAQ dichiara
    nel log, se si e' letta la run intera; se no l'arco del Trigger Time Tag."""
    if info and info["durata_s"] is not None and r["n"] == r["n_file"]:
        return info["durata_s"], "log"
    if len(r["ttt"]) > 1:
        passi = np.diff(r["ttt"].astype(np.int64)) % TTT_GIRO
        return float(passi.sum()) * TTT_NS * 1e-9, "TTT"
    return float("nan"), "?"


def cariche(r, c, segno, fin, dt):
    """(carica col piedistallo per evento, carica col piedistallo per blocco),
    in ADC*ns, positive nel verso del canale."""
    S = r["S_ch%d" % c]
    P = r["P_ch%d" % c].astype(np.float64)
    n_g = fin[c]["gi"][1] - fin[c]["gi"][0]
    # Un resto piu' corto di mezzo blocco va col blocco prima: la mediana di
    # pochi eventi ballerebbe.
    bordi = list(range(0, len(P), EVENTI_PER_BLOCCO)) + [len(P)]
    if len(bordi) > 2 and bordi[-1] - bordi[-2] < EVENTI_PER_BLOCCO // 2:
        del bordi[-2]
    P_blocco = np.concatenate([np.full(b - a, np.median(P[a:b]))
                               for a, b in zip(bordi[:-1], bordi[1:])])
    return ((segno * (S - P * n_g) * dt).astype(np.float32),
            (segno * (S - P_blocco * n_g) * dt).astype(np.float32))


def picco_a_zero(q, n_bin=2000):
    """Centro e larghezza del picco degli eventi senza impulso, dall'istogramma:
    la larghezza e' la FWHM / 2.355. Una deviazione standard, anche con
    l'esclusione a 3.5 sigma di padme.sigma_robusta, sui SiPM prende dentro la
    gobba degli impulsi veri, che sono un terzo degli eventi."""
    x = np.asarray(q, np.float64)
    h, e = np.histogram(x, bins=n_bin, range=tuple(np.percentile(x, [0.5, 99.5])))
    c = 0.5 * (e[1:] + e[:-1])
    k = int(np.argmax(h))
    meta = 0.5 * h[k]
    a, b = k, k
    while a > 0 and h[a] > meta:
        a -= 1
    while b < len(h) - 1 and h[b] > meta:
        b += 1
    # attraversamenti della meta' altezza, interpolati fra i due bin a cavallo
    xa = np.interp(meta, [h[a], h[a + 1]], [c[a], c[a + 1]])
    xb = np.interp(meta, [h[b], h[b - 1]], [c[b], c[b - 1]])
    dentro = (x > xa) & (x < xb)
    return float(x[dentro].mean()), float((xb - xa) / 2.3548)

In [ ]:
## === Le run scelte ===
FILE = trova_file(RUN)
NOMI_FILE = [efficienza.nome_run(p) for p in FILE]
_p, _lab, _s = etichette(NOMI_FILE)
NOME_USCITA = (NOMI_FILE[0] if len(FILE) == 1
               else "%s%s-%s%s_%drun" % (_p, _lab[0], _lab[-1], _s, len(FILE)))

print("%d run (dai log):" % len(FILE))
for p in FILE:
    info = leggi_log(p)
    if info is None:
        stato = "senza log: durata dal Trigger Time Tag, verso dal piedistallo"
    elif info["durata_s"] is None:
        stato = "in corso, o chiusa male: si legge quello che c'e'"
    else:
        stato = "%6s eventi in %5.0f s%s" % (info["eventi"] if info["eventi"] is not None else "?",
                                              info["durata_s"],
                                              ", collegamento perso" if info["collegamento_perso"] else "")
    print("  %-38s %s" % (os.path.basename(p), stato))

In [ ]:
## === Cancello e piedistallo, dalla prima run della lista ===
HDR_RIF, Y_RIF, DT = leggi_campione(FILE[0], EVENTI_PER_FINESTRE)
T_NS = np.arange(Y_RIF.shape[2]) * DT
INFO_RIF = leggi_log(FILE[0])
VERSI, FIN = {}, {}
print("Dai primi %d eventi di %s, a %.3f ns per campione:" % (len(Y_RIF), NOMI_FILE[0], DT))
for j, c in enumerate(CANALI):
    VERSI[c], come = verso_canale(c, INFO_RIF, Y_RIF[:, j, :])
    FIN[c] = finestre_canale(c, Y_RIF[:, j, :], VERSI[c], T_NS)
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    print("  ch%d %-4s impulsi %s (%s); cancello %.1f-%.1f ns (%s); piedistallo %.1f-%.1f ns, %s"
          % (c, RIVELATORE.get(c, ""), "positivi" if VERSI[c] > 0 else "negativi", come,
             g[0], g[1], "automatico" if FIN[c]["come"] == "automatic" else "a mano",
             T_NS[p[0]], T_NS[p[1] - 1], "per " + PIEDISTALLO.get(c, "evento")))
print("\nPer tenerli fissi qualunque sia la prima run, nella cella dei parametri:")
print("CANCELLO = {%s}" % ", ".join("%d: (%.1f, %.1f)" % (c, *FIN[c]["cancello_ns"]) for c in CANALI))

In [ ]:
## === Grafico: su cosa si sono decisi cancello e piedistallo ===
# Le forme d'onda con la loro polarita' vera; a destra dove cade il massimo
# degli stessi eventi: un impulso fuori dal cancello si vede li'.
os.makedirs(USCITA, exist_ok=True)
rng = np.random.default_rng(1)
fig, assi = plt.subplots(len(CANALI), 2, figsize=(14, 2.8 * len(CANALI)), squeeze=False,
                         gridspec_kw={"width_ratios": [1.7, 1]})
for (a, b), (j, c) in zip(assi, enumerate(CANALI)):
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    # La base e' la finestra del piedistallo, non la mediana della traccia: un
    # impulso di SiPM occupa meta' traccia e la mediana la alza di ~30 conteggi.
    y = Y_RIF[:, j, :].astype(np.float32)
    y -= y[:, p[0]:p[1]].mean(axis=1, keepdims=True)
    amp = (VERSI[c] * y).max(axis=1)
    alti = np.flatnonzero(amp >= np.percentile(amp, 80))
    for k in rng.choice(alti, size=min(30, len(alti)), replace=False):
        a.plot(T_NS, y[k], lw=0.4, color="#c3cbd8")
    a.plot(T_NS, y[alti].mean(axis=0), lw=1.4, color="#1f4e9c",
           label="mean of the 20%% highest (%d events), 30 of them in grey" % len(alti))
    a.axvspan(*g, color="#e8710a", alpha=0.18, label="gate %.0f-%.0f ns (%s)" % (g[0], g[1], FIN[c]["come"]))
    a.axvspan(T_NS[p[0]], T_NS[p[1] - 1], color="#888888", alpha=0.18,
              label="pedestal %.0f-%.0f ns" % (T_NS[p[0]], T_NS[p[1] - 1]))
    a.set_title("ch%d %s - first %d events of %s" % (c, RIVELATORE.get(c, ""), len(y), NOMI_FILE[0]),
                fontsize=9)
    a.set_xlabel("time [ns]")
    a.set_ylabel("counts above pedestal")
    a.legend(fontsize=7, loc="upper right")
    a.grid(alpha=0.25)

    b.hist(T_NS[np.argmax(VERSI[c] * y[alti], axis=1)],
           bins=np.arange(0, T_NS[-1] + 8 * DT, 8 * DT), color="#1f4e9c")
    b.axvspan(*g, color="#e8710a", alpha=0.18)
    b.set_title("ch%d - time of the maximum, same events" % c, fontsize=9)
    b.set_xlabel("time [ns]")
    b.set_ylabel("events")
    b.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(USCITA, "carica_run_%s_finestre.png" % NOME_USCITA), dpi=110)
plt.show()

In [ ]:
## === Lettura delle run ===
RISULTATI = []
inizio = time.time()
for k, path in enumerate(FILE):
    t1 = time.time()
    nome = efficienza.nome_run(path)
    info = leggi_log(path)
    controlla_versi(info, nome)
    r, da_cache = leggi_run(path, info, FIN, HDR_RIF)
    if r["n"] == 0:
        print("[%2d/%d] %s: nessun evento, saltata" % (k + 1, len(FILE), nome))
        continue
    d, fonte = durata(r, info)
    note = []
    if info is None:
        note.append("senza log")
    else:
        if info["durata_s"] is None:
            note.append("in corso o chiusa male")
        if info["collegamento_perso"]:
            note.append("collegamento perso")
    if r["n"] < r["n_file"]:
        note.append("letti %d di %d" % (r["n"], r["n_file"]))
    RISULTATI.append({"nome": nome, "path": path, "r": r, "durata_s": d, "fonte": fonte,
                      "drs4": info["drs4"] if info else "?", "note": note})
    print("[%2d/%d] %-34s %7d eventi in %6.0f s (%s), %5.1f Hz   %-7s %s"
          % (k + 1, len(FILE), nome, r["n"], d, fonte, r["n"] / d if d else float("nan"),
             "cache" if da_cache else "%.0f s" % (time.time() - t1), ", ".join(note)))
print("lette in %.0f s" % (time.time() - inizio))

NOMI = [x["nome"] for x in RISULTATI]
DURATE = np.array([x["durata_s"] for x in RISULTATI])
RUN_IDX = np.concatenate([np.full(x["r"]["n"], i, np.int32) for i, x in enumerate(RISULTATI)])
EVENTO = np.concatenate([np.arange(x["r"]["n"], dtype=np.int32) for x in RISULTATI])
Q, SAT, ZERO = {}, {}, {}
for c in CANALI:
    coppie = [cariche(x["r"], c, VERSI[c], FIN, DT) for x in RISULTATI]
    Q[c] = {"evento": np.concatenate([e for e, _ in coppie]),
            "blocco": np.concatenate([b for _, b in coppie])}
    SAT[c] = np.concatenate([x["r"]["sat_ch%d" % c] for x in RISULTATI])
    ZERO[c] = {m: picco_a_zero(Q[c][m]) for m in Q[c]}

print("\n%-6s %-5s %-14s %-14s %-7s %8s %7s   picco a zero, centro +- FWHM/2.355 [ADC*ns]: per evento / per blocco"
      % ("canale", "", "cancello [ns]", "piedist. [ns]", "scelto", "eventi", "saturi"))
for c in CANALI:
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    print("ch%-4d %-5s %5.0f - %-6.0f %5.0f - %-6.0f %-7s %8d %7d   %+.0f +- %.0f / %+.0f +- %.0f"
          % (c, RIVELATORE.get(c, ""), g[0], g[1], T_NS[p[0]], T_NS[p[1] - 1],
             PIEDISTALLO.get(c, "evento"), len(SAT[c]), SAT[c].sum(),
             *ZERO[c]["evento"], *ZERO[c]["blocco"]))

In [ ]:
# ============================================================
# GRAFICI
# ============================================================
# Asse delle cariche per canale, in ADC*ns: XLIM = {20: (-2000, 40000)}.
# Senza, dallo 0.05 al 99.95 percentile; il titolo dice quanti eventi restano fuori.
XLIM = {}
N_BIN = 200
MAX_CURVE = 10      # nel confronto fra le run: oltre, run consecutive sommate a gruppi

ALTRO = {"evento": "blocco", "blocco": "evento"}
DESCR = {"evento": "pedestal per event",
         "blocco": "pedestal per block of %d events" % EVENTI_PER_BLOCCO}


def intervallo(c, q):
    if c in XLIM:
        return XLIM[c]
    lo, hi = np.percentile(q, [0.05, 99.95])
    return lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo)


def pannelli(titolo):
    """Una griglia a due colonne riempita per colonne: con i cinque canali del
    LNGS i SiPM stanno a sinistra, i PMT a destra, e il sesto riquadro resta
    libero per la legenda o il riepilogo."""
    nr = (len(CANALI) + 1) // 2
    fig, assi = plt.subplots(nr, 2, figsize=(14, 3.6 * nr), squeeze=False)
    fig.suptitle(titolo, fontsize=11)
    return fig, [assi[i % nr, i // nr] for i in range(len(CANALI))], \
        (assi[-1, -1] if len(CANALI) % 2 else None)

In [ ]:
## === Distribuzioni di carica ===
fig, assi, libero = pannelli("%s - charge per channel" % (
    NOMI[0] if len(NOMI) == 1 else "%d runs, %s ... %s" % (len(NOMI), NOMI[0], NOMI[-1])))
for a, c in zip(assi, CANALI):
    scelto = PIEDISTALLO.get(c, "evento")
    q, q2, sat = Q[c][scelto], Q[c][ALTRO[scelto]], SAT[c]
    lo, hi = intervallo(c, q)
    bins = np.linspace(lo, hi, N_BIN + 1)
    fuori = int(((q < lo) | (q > hi)).sum())
    a.hist(q2, bins, histtype="step", color="#9a9a9a", lw=0.9,
           label="%s  (zero peak: σ = %.0f)" % (DESCR[ALTRO[scelto]], ZERO[c][ALTRO[scelto]][1]))
    a.hist(q, bins, histtype="step", color="#1f4e9c", lw=1.4,
           label="%s  (zero peak: σ = %.0f)" % (DESCR[scelto], ZERO[c][scelto][1]))
    if sat.any():
        a.hist(q[sat], bins, color="#e8710a", alpha=0.5,
               label="saturated: %d events (charge is a lower limit)" % sat.sum())
    g = FIN[c]["cancello_ns"]
    a.set_title("ch%d %s - gate %.0f-%.0f ns, %d events%s" % (
        c, RIVELATORE.get(c, ""), g[0], g[1], len(q), ", %d outside the axis" % fuori if fuori else ""),
        fontsize=9.5)
    a.set_yscale("log")
    a.set_xlim(lo, hi)
    a.set_xlabel("charge [ADC·ns]   (1 pC = 204.8 ADC·ns)")
    a.set_ylabel("events / %.0f ADC·ns" % (bins[1] - bins[0]))
    a.legend(fontsize=7.5, frameon=False)
    a.grid(alpha=0.25)
if libero is not None:
    libero.axis("off")
    drs4 = sorted({x["drs4"] for x in RISULTATI})
    righe = ["%d run%s, %d events, %.2f h of acquisition (%.1f Hz)" % (
                 len(NOMI), "s" if len(NOMI) > 1 else "", len(RUN_IDX), DURATE.sum() / 3600,
                 len(RUN_IDX) / DURATE.sum()),
             "first: %s" % NOMI[0]] + (["last:  %s" % NOMI[-1]] if len(NOMI) > 1 else []) + [
             "gates and pedestals from the first %d events of the first run" % len(Y_RIF),
             "sampling %.3f ns; CAEN DRS4 correction %s; no PADME" % (DT, "/".join(drs4))]
    libero.text(0.0, 0.95, "\n".join(righe), va="top", fontsize=9, transform=libero.transAxes)
fig.tight_layout()
fig.savefig(os.path.join(USCITA, "carica_run_%s.png" % NOME_USCITA), dpi=110)
plt.show()

In [ ]:
## === Confronto fra le run ===
# Ogni curva e' una run, o un gruppo di run consecutive se sono piu' di
# MAX_CURVE, divisa per il suo tempo di acquisizione: run di durata diversa
# si confrontano in rate, e una deriva del guadagno sposta le curve.
if len(NOMI) < 2:
    print("Una run sola: niente confronto.")
else:
    gruppi = np.array_split(np.arange(len(NOMI)), min(MAX_CURVE, len(NOMI)))
    _, lab, _ = etichette(NOMI)
    colori = plt.cm.viridis(np.linspace(0, 0.92, len(gruppi)))
    fig, assi, libero = pannelli("runs compared: rate per bin, each run divided by its acquisition time")
    for a, c in zip(assi, CANALI):
        scelto = PIEDISTALLO.get(c, "evento")
        q = Q[c][scelto]
        lo, hi = intervallo(c, q)
        bins = np.linspace(lo, hi, N_BIN + 1)
        for gr, col in zip(gruppi, colori):
            h, _ = np.histogram(q[np.isin(RUN_IDX, gr)], bins)
            a.stairs(h / DURATE[gr].sum(), bins, color=col, lw=1.0,
                     label=lab[gr[0]] if len(gr) == 1 else "%s - %s" % (lab[gr[0]], lab[gr[-1]]))
        a.set_title("ch%d %s - %s" % (c, RIVELATORE.get(c, ""), DESCR[scelto]), fontsize=9.5)
        a.set_yscale("log")
        a.set_xlim(lo, hi)
        a.set_xlabel("charge [ADC·ns]")
        a.set_ylabel("rate / %.0f ADC·ns [Hz]" % (bins[1] - bins[0]))
        a.grid(alpha=0.25)
    if libero is not None:
        libero.axis("off")
        libero.legend(*assi[0].get_legend_handles_labels(), loc="upper left", fontsize=9,
                      title="run" if all(len(gr) == 1 for gr in gruppi) else "runs (consecutive runs summed)",
                      frameon=False)
    else:
        assi[0].legend(fontsize=7.5, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(USCITA, "carica_run_%s_confronto.png" % NOME_USCITA), dpi=110)
    plt.show()

In [ ]:
## === Salvataggio ===
# Le cariche in DATI, fuori da git: 50 run sono ~150 MB. In measurements/
# solo la descrizione, che dice anche dove sono.
os.makedirs(DATI, exist_ok=True)
npz = os.path.join(DATI, "carica_run_%s.npz" % NOME_USCITA)
base = os.path.join(USCITA, "carica_run_" + NOME_USCITA)
extra = {}
if all(len(x["r"]["ttt"]) == x["r"]["n"] for x in RISULTATI):
    extra["event_counter"] = np.concatenate([x["r"]["event_counter"] for x in RISULTATI])
    extra["trigger_time_tag"] = np.concatenate([x["r"]["ttt"] for x in RISULTATI])
meta = {
    "run": NOMI, "file": [x["path"] for x in RISULTATI],
    "eventi": [int(x["r"]["n"]) for x in RISULTATI],
    "durata_s": [float(d) for d in DURATE], "durata_da": [x["fonte"] for x in RISULTATI],
    "note": [x["note"] for x in RISULTATI],
    "dt_ns": DT, "unita": "ADC*ns, positiva nel verso del canale (1 pC = 204.8)",
    "canali": {str(c): {"rivelatore": RIVELATORE.get(c, ""), "verso": int(VERSI[c]),
                        "cancello_ns": [float(x) for x in FIN[c]["cancello_ns"]],
                        "cancello": FIN[c]["come"],
                        "piedistallo_ns": [float(T_NS[FIN[c]["pi"][0]]), float(T_NS[FIN[c]["pi"][1] - 1])],
                        "piedistallo_scelto": PIEDISTALLO.get(c, "evento"),
                        "picco_a_zero": {m: list(ZERO[c][m]) for m in ZERO[c]}}
               for c in CANALI},
    "eventi_per_blocco": EVENTI_PER_BLOCCO,
    "finestre_da": "primi %d eventi di %s" % (len(Y_RIF), NOMI_FILE[0]),
    "padme": "nessuna", "correzione_drs4": [x["drs4"] for x in RISULTATI],
    "creato": time.strftime("%Y-%m-%d %H:%M:%S"), "da": "tools/Distribuzioni_carica.ipynb",
    "cariche": npz,
}
np.savez_compressed(npz, run=np.array(NOMI), durata_s=DURATE, run_idx=RUN_IDX,
                    evento=EVENTO, **extra,
                    **{"q_evento_ch%d" % c: Q[c]["evento"] for c in CANALI},
                    **{"q_blocco_ch%d" % c: Q[c]["blocco"] for c in CANALI},
                    **{"saturo_ch%d" % c: SAT[c] for c in CANALI},
                    meta=json.dumps(meta))
with open(base + ".json", "w") as f:
    json.dump(meta, f, indent=1)
print("Cariche evento per evento: %s\nDescrizione e grafici:     %s.json, %s*.png" % (npz, base, base))